# finrag evaluations on a free GPU (Kaggle or Colab)

Runs the same commands as the GitHub workflows `extraction-eval` and `heldout-eval`, but on a free GPU, so they take minutes instead of hours.

- **Kaggle**: Settings > Accelerator > GPU T4 x2 (or P100), and Internet on (needs a verified phone number).
- **Colab**: Runtime > Change runtime type > T4 GPU.

The documents are the 45 sheets frozen in `eval/fixtures/fr-interieur/docs` (Licence Ouverte 2.0, DILA). The model is checked against its pinned digest before any run. A 16 GB GPU holds `llama-3b` and `qwen3-4b`; `mistral-24b` needs 24 GB.

In [ ]:
MODEL = "llama-3b"   # or "qwen3-4b"
BRANCH = "main"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Ollama 0.12.6, the version deploy/k8s and the GitHub workflows run, from its official release
!curl -fsSL -o /tmp/ollama.tgz https://github.com/ollama/ollama/releases/download/v0.12.6/ollama-linux-amd64.tgz
!tar -xzf /tmp/ollama.tgz -C /usr/local && rm /tmp/ollama.tgz
import subprocess, time, urllib.request
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):  # wait until the server answers
    try:
        print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode()); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start")


In [ ]:
# The code, at the chosen branch, with its locked dependencies
!rm -rf finrag && git clone -q --branch $BRANCH https://github.com/HadiJamalAhmad/finrag.git
%cd finrag
!pip -q install uv && uv sync --frozen -q && git log --oneline -1

In [ ]:
import os
os.environ.update({
    "OLLAMA_URL": "http://localhost:11434", "FINRAG_MODEL": MODEL,
    "FINRAG_DATA": "eval/fixtures/fr-interieur", "PYTHONUNBUFFERED": "1",
    # frozen settings of the French documents mode (deploy/k8s/overlays/fr)
    "FINRAG_DOMAIN": "documents",
    "FINRAG_EMBED_MODEL": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "FINRAG_COLLECTION": "docs_fr", "FINRAG_MIN_SCORE": "0.61", "FINRAG_TOP_K": "3",
})
!uv run finrag pull-model

In [ ]:
# 1. Extraction: every field must quote the sheet sentence that proves it
!uv run finrag eval-extract eval/suites/fr-extraction.jsonl --cache eval/reports/extraction-cache-$MODEL --out eval/reports/extraction-$MODEL.json

In [ ]:
# 2. Held-out questions, settings frozen beforehand
!uv run finrag ingest > /dev/null
!uv run finrag eval eval/suites/fr-interieur-heldout.jsonl --out eval/reports/fr-heldout-$MODEL.json

In [ ]:
# The reports, to download (Kaggle: Output tab; Colab: Files panel) and commit
!cd eval/reports && zip -qr /tmp/finrag-reports-$MODEL.zip . && ls -la /tmp/finrag-reports-$MODEL.zip
!ollama stop $(ollama ps | awk 'NR==2{print $1}') 2>/dev/null; true